# Lesson 02 · Activation functions: from the step switch to SwiGLU, on a real MNIST network

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/02-ham-kich-hoat-en.ipynb)

This notebook accompanies [Lesson 02 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-kich-hoat/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/02-ham-kich-hoat.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
After the weighted sum $z = xW + b$, each neuron passes $z$ through an **activation function** $f$. We write these
functions and their derivatives ourselves in numpy, then put each one into the course's 784 → 128 → 64 → 10 MNIST
network to see what $f$ decides: whether the network is really “deep”, whether it can learn, and how fast it learns.

Section 0 sets out all of the lesson's theory first, without running anything; the sections after it rebuild each
idea in numpy and check it with numbers.

**What you'll do:**
1. Write six activation functions and their derivatives $f'$, and check them with finite differences.
2. See numerically that without $f$, three layers collapse into one matrix, and why the step function gives no
   gradient.
3. Measure the saturation regions of sigmoid and tanh, then pass the real $z$ of the MNIST network (retrained,
   97.27%) through each function.
4. Retrain the same network with each function, three seeds per function, and read the results as an experiment.
5. Build a SwiGLU layer, the “gated MLP” of Llama 3, Qwen3 and DeepSeek-V3, and count its parameters.

**Background:** [Lesson 01 · Perceptron](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/perceptron/)
(one neuron, $z = xW + b$, the step function); derivatives and the chain rule (section 0 uses them; Lessons 05–06
develop them fully). **Run time:** under a minute on a CPU, including the first MNIST download (~11 MB).

The notation follows the *Neural Computation* chapter of [MLSysBook](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html).
Every number in the notebook is computed by the cells below. The numbers that the demo and the reel also show
($\sigma'(0) = 0.25$, the reference network's 97.27%…) all match; the `assert` cells check that.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 0 · Theory before code

This section gathers all of the lesson's theory in one place, and runs no cells yet. Sections 1–12 rebuild each idea
in numpy on the real MNIST network; the table at the end of this section says where each one is checked. Notation,
following MLSysBook: $x$, $z$, $h$ are row vectors; layer $\ell$ has a weight matrix $W_\ell$ of size
$n_{\ell-1} \times n_\ell$ and a bias $b_\ell$; $f$ is applied to each number separately; $\odot$ is element-wise
multiplication.

### 0.1 · The activation function, and why its derivative matters

A layer computes the weighted sum $z = xW + b$, then passes each number of $z$ through the same **activation
function** $f$: $h = f(z)$. The course's network, MLSysBook's 784 → 128 → 64 → 10 MLP, stacks three such layers:

$$h_0 = x, \qquad z_\ell = h_{\ell-1}W_\ell + b_\ell, \qquad h_\ell = f(z_\ell) \;\; (\ell = 1, 2), \qquad z_3 = h_2W_3 + b_3$$

The last layer gives the 10 logits $z_3$ and has no $f$; softmax and the cross-entropy loss $L$ are the subject of
Lesson 04.

The network learns by gradient descent (Lesson 05), so it needs the derivative of $L$ with respect to every weight.
Let $\delta_\ell = \partial L / \partial z_\ell$ (a row vector). For a hidden layer $\ell$ and one image,
$z_{\ell+1,j} = \sum_i f(z_{\ell,i})\,W_{\ell+1,ij} + b_{\ell+1,j}$, and $z_{\ell,i}$ affects $L$ only through
$z_{\ell+1}$, so the chain rule gives

$$\frac{\partial L}{\partial z_{\ell,i}} = \sum_j \frac{\partial L}{\partial z_{\ell+1,j}}\, W_{\ell+1,ij}\, f'(z_{\ell,i}) \quad\Longrightarrow\quad \delta_\ell = \big(\delta_{\ell+1} W_{\ell+1}^\top\big) \odot f'(z_\ell)$$

and, since $z_{\ell,j} = \sum_i h_{\ell-1,i}\, W_{\ell,ij} + b_{\ell,j}$,

$$\frac{\partial L}{\partial W_\ell} = h_{\ell-1}^\top\, \delta_\ell, \qquad \frac{\partial L}{\partial b_\ell} = \delta_\ell.$$

For a mini-batch of many images, the row vectors stack into matrices and the gradient is the sum (or the mean) over
the images. Lesson 06 derives backpropagation in full; here we only need the place where $f$ appears: each time the
error signal $\delta$ travels backward through a hidden layer, each of its components is multiplied by the $f'$ of
that very neuron. Since $f$ acts on each number separately, this is an element-wise product, not a matrix product.
So for each activation function, the lesson asks two questions: is $f$ nonlinear (section 0.2), and how large is
$f'$ (the rest of this section).

### 0.2 · Without $f$: all affine layers collapse into one

Drop $f$ (that is, $f(z) = z$), and each layer is an **affine map** $x \mapsto xW + b$. Compose two such layers:

$$(xW_1 + b_1)\,W_2 + b_2 = x\,(W_1W_2) + (b_1W_2 + b_2),$$

which is still an affine map, with matrix $W_1W_2$ and bias $b_1W_2 + b_2$. Doing it once more (and, by induction,
for any number of layers), the network's three layers collapse into

$$z_3 = x\,\underbrace{W_1W_2W_3}_{784 \times 10} + \underbrace{(b_1W_2 + b_2)W_3 + b_3}_{1 \times 10}.$$

So a deep network without $f$ cannot represent any function that a single affine layer 784 → 10 cannot, even though
it costs $784 \cdot 128 + 128 \cdot 64 + 64 \cdot 10$ multiply-accumulate (MAC) operations per image instead of
$784 \cdot 10$. The collapse only needs each layer to be affine, so it still holds for any $f(z) = az + c$: $f$ must
be **nonlinear** for depth to mean anything. With a nonlinear $f$ between the layers, $f(xW_1 + b_1)$ is in general
no longer an affine function of $x$, and no affine map can replace the whole network.

### 0.3 · The step function: nonlinear, but with no slope

$$f(z) = \begin{cases} 1 & z \ge 0 \\ 0 & z < 0 \end{cases}$$

The perceptron's function (Lesson 01) is nonlinear, so it doesn't have the flaw of section 0.2. Its flaw is a
different one: it is constant on each half of the axis, so $f'(z) = 0$ at every $z \ne 0$; at $z = 0$ it jumps from
0 to 1 and has no derivative (the code in section 3 sets $f' = 0$ everywhere). Put this into the formula of
section 0.1: in a step layer, $\delta_\ell = (\delta_{\ell+1} W_{\ell+1}^\top) \odot 0 = 0$ for every image, so
$\partial L/\partial W_\ell = h_{\ell-1}^\top\,\delta_\ell = 0$ and $\partial L/\partial b_\ell = 0$. In the lesson's
network, both hidden layers use $f$, so with the step function $\delta_1 = \delta_2 = 0$: the gradients of
$W_1, b_1, W_2, b_2$ are 0, and gradient descent has no direction in which to adjust them. Lesson 01's Rosenblatt
rule doesn't run into this problem because it uses no derivative: it compares the output with the label directly,
and there is only one layer.

### 0.4 · Sigmoid and tanh: a smooth switch

**Sigmoid** (the logistic function) goes from 0 to 1 like the step function, but without a break:

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad \sigma'(z) = \frac{e^{-z}}{(1 + e^{-z})^2} = \sigma(z)\,\big(1 - \sigma(z)\big) = \frac{1}{4\cosh^2(z/2)}.$$

The second equality holds because $e^{-z}/(1 + e^{-z}) = 1 - \sigma(z)$; the third multiplies numerator and
denominator by $e^{z}$, giving $1/(e^{z/2} + e^{-z/2})^2$. The output lies in $(0, 1)$ and
$\sigma(-z) = 1 - \sigma(z)$, so $\sigma'$ is an even function. With $s = \sigma(z)$,
$s(1 - s) = \tfrac14 - (s - \tfrac12)^2 \le \tfrac14$, with equality only when $s = \tfrac12$, that is $z = 0$:

$$\sigma'(z) \le \sigma'(0) = \tfrac14.$$

Sigmoid is a smoothed step function. For $T > 0$, as $T \to 0$, $\sigma(z/T) \to 1$ if $z > 0$ (since
$e^{-z/T} \to 0$) and $\sigma(z/T) \to 0$ if $z < 0$: $\sigma(z/T)$ approaches the step function at every $z \ne 0$
(at $z = 0$ it always equals $\tfrac12$). Its slope at 0 is $\sigma'(0)/T = 1/(4T)$: the closer it gets to the step
function, the steeper but the narrower it is.

**Tanh** is a rescaled sigmoid:

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}} = 2\sigma(2z) - 1, \qquad \tanh'(z) = 4\sigma'(2z) = \frac{1}{\cosh^2 z} = 1 - \tanh^2(z).$$

Proof: $2\sigma(2z) - 1 = (1 - e^{-2z})/(1 + e^{-2z})$; then multiply numerator and denominator by $e^{z}$.
Differentiating both sides gives $\tanh'(z) = 4\sigma'(2z)$. As a picture: $\sigma(2z)$ squeezes sigmoid to half its
width, the factor 2 makes it twice as tall, and $-1$ shifts it down. So tanh has its output in $(-1, 1)$, is an odd
function (symmetric around 0, while sigmoid is always positive), and $\tanh'(z) \le \tanh'(0) = 1$, 4 times
$\sigma'(0)$. Read backward (with $z/2$ in place of $z$), the identity gives
$\sigma(z) = \tfrac12\,(1 + \tanh(z/2))$; section 3 computes sigmoid in this form so that it doesn't overflow.

### 0.5 · Saturation: derivatives that die out exponentially

When $|z|$ is large, both curves are almost horizontal. More precisely: since $\sigma'$ is even,
$\sigma'(z) = e^{-|z|}/(1 + e^{-|z|})^2$, and the denominator lies between 1 and 4, so

$$\tfrac14\, e^{-|z|} \le \sigma'(z) \le e^{-|z|}, \qquad e^{-2|z|} \le \tanh'(z) \le 4\,e^{-2|z|}$$

(the bounds for tanh follow from $\tanh'(z) = 4\sigma'(2z)$). For large $|z|$ the denominator tends to 1 and the
upper bound is almost tight: each time $|z|$ grows by 1, $\sigma'$ shrinks by a factor of almost $e$ and $\tanh'$ by
almost $e^2$. A value of $z$ (of one neuron, for one input) is **saturated** when $f'(z)$ is less than one tenth of
its largest value $f'(0)$. That threshold has an exact solution:

$$\sigma'(z) < \tfrac{1}{40} \iff \cosh^2(z/2) > 10 \iff |z| > 2\operatorname{arccosh}\sqrt{10} = 2\ln\big(3 + \sqrt{10}\big) \approx 3.64$$

$$\tanh'(z) < \tfrac{1}{10} \iff \cosh^2 z > 10 \iff |z| > \operatorname{arccosh}\sqrt{10} = \ln\big(3 + \sqrt{10}\big) \approx 1.82$$

since $\operatorname{arccosh} y = \ln\big(y + \sqrt{y^2 - 1}\big)$. Tanh's threshold is exactly half of sigmoid's,
again because $\tanh'(z) = 4\sigma'(2z)$. Saturation is a matter of each (input, neuron) pair: the same neuron can be
saturated for one input and not for another.

**Across many layers.** Unroll the formula $\delta_\ell = (\delta_{\ell+1} W_{\ell+1}^\top) \odot f'(z_\ell)$ of
section 0.1 through $L$ hidden layers (here, as in section 6, $L$ is the number of layers, not the loss): the first
layer's $\delta$ is a sum over the paths backward through the network, and each term is the product of the output
layer's $\delta$, the weights along the path, and $L$ factors $f'$, one per hidden layer. With sigmoid, each factor
is at most $\tfrac14$, so the $f'$ part of each term is at most $(1/4)^L$, for example
$(1/4)^{20} \approx 9.1 \cdot 10^{-13}$ (the number MLSysBook gives). That is the best case, when every $z = 0$; a
saturated neuron contributes a factor of at most $e^{-|z|}$. The weights can make the signal larger or smaller, so
this bound is only about the $f'$ part. The signal dying out across the layers is the **vanishing gradient**
problem (Lesson 09).

### 0.6 · ReLU, its derivative, and dead neurons

$$\operatorname{ReLU}(z) = \max(0, z), \qquad \operatorname{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & z < 0 \end{cases}$$

ReLU keeps the positive part as it is and clips the negative part to 0. At $z = 0$ it has a corner (slope 0 on the
left, 1 on the right), so it has no derivative there; the code in section 3 sets $f'(0) = 0$, as PyTorch does, and
that single value only matters for inputs whose $z$ is exactly 0. Three consequences:
- **No saturation on the positive side.** When $z > 0$, $f' = 1$ however large $z$ is, so by section 0.1, $\delta$
  passes through a neuron that is on without shrinking; its size is then decided by the matrices $W$ alone.
- **Cheap.** One comparison with 0. Sigmoid, tanh, GELU and SiLU all need the exponential function, which, as
  MLSysBook notes, hardware computes with special instructions, lookup tables or polynomials.
- **Fully off when $z \le 0$.** For an input with $z \le 0$, the neuron outputs exactly 0 and has $f' = 0$: that
  input sends no gradient through this neuron.

**Dead neurons.** Call neuron $j$ of hidden layer $\ell$ **dead** when $z_{\ell,j} \le 0$ for *every* training
image. Then, for every image, $h_{\ell,j} = 0$ and $f'(z_{\ell,j}) = 0$, so by section 0.1, $\delta_{\ell,j} = 0$.
Two things follow, for every image and therefore for every mini-batch: the gradient of its incoming weights (column
$j$ of $W_\ell$, equal to $h_{\ell-1}^\top\,\delta_{\ell,j}$) and of its bias $b_{\ell,j}$ is 0; and the gradient of
its outgoing weights (row $j$ of $W_{\ell+1}$, equal to $h_{\ell,j}\,\delta_{\ell+1}$) is 0 too. With plain SGD, as
in this lesson, those weights never change again. In layer 1 the inputs are the images themselves, which don't
change, so $z_{1,j}$ stays $\le 0$ for every training image: the neuron never comes back to life. (In a deeper layer,
the input $h_{\ell-1}$ still changes as the earlier layers learn, so in principle the neuron could still come back.)
A step that is too large can push a neuron into that state. Dead is not the same as off: a neuron that outputs 0 for
one image still learns from the images that switch it on. LeakyReLU, with a small slope (for example $0.01$) on the
negative side, never has $f' = 0$, so it has no dead neurons. Nair and Hinton (2010) used ReLU for restricted
Boltzmann machines; since then it has become the default choice for many deep networks.

### 0.7 · GELU and SiLU: ReLU, smoothed

$$\operatorname{GELU}(z) = z\,\Phi(z), \qquad \operatorname{SiLU}(z) = z\,\sigma(z)$$

$\Phi(z) = \tfrac12\big(1 + \operatorname{erf}(z/\sqrt2)\big)$ is the probability that a standard normal variable
$\mathcal N(0, 1)$ is less than $z$. Both functions are $z$ times a **soft gate** that rises from 0 to 1: as
$z \to +\infty$, the gate tends to 1 and $f(z) \approx z$, like ReLU; as $z \to -\infty$, $f(z) \to 0$. In between
they are smooth, and since $z < 0$ times a positive number is negative, both are negative on the whole negative
half-axis: they are not monotonic, dipping to a lowest point and then coming back to 0. Hendrycks and Gimpel (2016)
named both; Elfwing et al. (2017) used SiLU in reinforcement learning; Ramachandran et al. (2017) rediscovered it
with an automated search and called it Swish, $z\,\sigma(\beta z)$, which with $\beta = 1$ is exactly SiLU. Their
derivatives, with $\varphi = \Phi'$ the density of the standard normal distribution:

$$\operatorname{GELU}'(z) = \Phi(z) + z\,\varphi(z), \qquad \operatorname{SiLU}'(z) = \sigma(z)\,\big(1 + z\,(1 - \sigma(z))\big)$$

Both equal $\tfrac12$ at 0.

**SiLU's lowest point.** Since $\sigma > 0$, $\operatorname{SiLU}'(z) = 0 \iff 1 + z\,(1 - \sigma(z)) = 0$, and
$1 - \sigma(z) = 1/(1 + e^{z})$, so $z = -(1 + e^{z})$. Let $u = -1 - z$: $u = e^{-1-u}$, that is $u\,e^{u} = 1/e$,
so $u = W(1/e)$, where $W$ is the Lambert function (the solution of $u\,e^u = y$). This solution is unique, because
$1 + z/(1 + e^z)$ increases on the negative half-axis and is positive on the positive one. At $z^* = -1 - W(1/e)$
we have $1 + e^{z^*} = -z^*$, so $\sigma(z^*) = 1 + 1/z^*$ and

$$\min_z \operatorname{SiLU}(z) = z^*\sigma(z^*) = z^* + 1 = -W(1/e) \approx -0.2785.$$

The same constant appears on the positive side: $\operatorname{SiLU}'(z) - 1 = (1 - \sigma(z))\,(z\,\sigma(z) - 1)$,
so $\operatorname{SiLU}'(z) > 1$ exactly where $\operatorname{SiLU}(z) > 1$, that is for $z > 1 + W(1/e)$ (let
$u = z - 1$ as above). SiLU's slope therefore rises above 1, then tends to 1 from above as $z \to \infty$. GELU has
no closed-form solution like this; section 7 finds its lowest point on a grid.

**Two ways to compute GELU.** Hendrycks and Gimpel give two approximations of $\Phi$:
$\tfrac12\big(1 + \tanh[\sqrt{2/\pi}\,(z + 0.044715\,z^3)]\big)$, the form that GPT-2 and the demo use, and
$\sigma(1.702\,z)$. Both equal $\tfrac12$ at 0, like $\Phi$; the tanh form also has the same slope as $\Phi$ there,
$\tfrac12\sqrt{2/\pi} = 1/\sqrt{2\pi} = \varphi(0)$. Section 7 measures the error of each form.

### 0.8 · Gated MLPs: GLU and SwiGLU

Every Transformer block (Lesson 17) has a two-layer MLP, called the FFN:
$\operatorname{FFN}(x) = f(xW_1 + b_1)\,W_2 + b_2$, with $W_1$ of size $d \times d_{ff}$, usually $d_{ff} = 4d$;
GPT-2 and BERT use $f$ = GELU. A **gated linear unit** (GLU, Dauphin et al., 2016) uses two parallel projections and
multiplies them element-wise, $\sigma(xW) \odot xV$. Shazeer (2020) tried many variants; the one that uses SiLU is
SwiGLU, without biases:

$$\operatorname{FFN}_{\text{SwiGLU}}(x) = \big(\operatorname{SiLU}(xW) \odot xV\big)\,W_2$$

Hidden neuron $i$ receives $u = (xW)_i$ and $v = (xV)_i$ and outputs $\operatorname{SiLU}(u)\,v = \sigma(u)\cdot uv$:
the number $\sigma(u) \in (0, 1)$ is a **gate** that decides how much of $uv$ gets through, and since $W$ and $V$ are
both learned, the network also learns when to open the gate. Sigmoid lives on here, inside SiLU.

**Parameter count.** Ignoring biases, an FFN has $2\,d\,d_{ff}$ weights ($W_1$ and $W_2$); SwiGLU has three
matrices, $3\,d\,d_{ff}'$ weights. The two counts are equal when $d_{ff}' = \tfrac23 d_{ff}$, that is $\tfrac83 d$ if
$d_{ff} = 4d$. Shazeer compares them exactly like that, so the two options have the same number of parameters and
the same number of multiply-accumulate operations in their matrix multiplications.

**The gradient through the gate.** With $g = \operatorname{SiLU}(u)$ and $h = g \odot v$, the chain rule gives

$$\frac{\partial L}{\partial v} = \frac{\partial L}{\partial h} \odot g, \qquad \frac{\partial L}{\partial u} = \frac{\partial L}{\partial h} \odot v \odot \operatorname{SiLU}'(u),$$

then $\partial L/\partial W = x^\top\, \partial L/\partial u$ and
$\partial L/\partial V = x^\top\, \partial L/\partial v$: the signal splits into two branches, and only the branch
through the gate is multiplied by the derivative of an activation function. SwiGLU's gain is an empirical result;
Shazeer offers no theoretical explanation.

### Where each piece of theory is checked

| Theory | Checked with numbers in |
|---|---|
| 0.1 $f'$ in backpropagation, $\delta_\ell = (\delta_{\ell+1}W_{\ell+1}^\top) \odot f'(z_\ell)$ | section 3 (each $f'$ matches finite differences), section 10 (the SwiGLU network's hand-written gradient matches finite differences) |
| 0.2 affine layers collapse into one | section 4 (the merged matrix; with ReLU there is no merge), section 12 (XOR) |
| 0.3 the step function, $f' = 0$ | section 5 (nudging a bias), section 9 ($W_1$, $W_2$ stay put) |
| 0.4 sigmoid, tanh and their derivatives | section 3 ($f(0)$, $f'(0)$, the largest $f'$), section 5 ($\sigma(z/T)$), section 6 ($\tanh(z) = 2\sigma(2z) - 1$) |
| 0.5 saturation, $e^{-\lvert z \rvert}/4 \le \sigma'(z) \le e^{-\lvert z \rvert}$, the bound $(1/4)^L$ | section 6, section 8 (the network's real $z$), section 11 (tanh with initialization × 10) |
| 0.6 ReLU and dead neurons | section 7, section 8 (0 for one image but not dead), section 11 |
| 0.7 GELU, SiLU, the lowest point $-W(1/e)$ | section 3, section 7 |
| 0.8 SwiGLU: parameter count, the gradient through the gate | section 10 |

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign. `sci()` prints very small numbers
in the form $9.1 \cdot 10^{-13}$.

In [ ]:
import gzip
import hashlib
import itertools
import math
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (solid/dashed lines, shapes, labels)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    return f"{x:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig):
    """Put the tick labels of every linear axis (colorbars included) in the English format; call right before
    plt.show()."""
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
            if axis.get_scale() == "linear" and isinstance(axis.get_major_formatter(), ScalarFormatter):
                axis.set_major_formatter(FuncFormatter(tick))


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """A very small or very large number: 9.1·10⁻¹³."""
    if x == 0:
        return "0"
    m, e = f"{x:.{digits}e}".split("e")
    return f"{num(float(m), digits)}·10{str(int(e)).translate(SUP)}"


print(num(13007, 0), num(0.1, 1), pct(0.9727), sci(0.25**20))

## 2 · The MNIST data

The same data as in Lesson 01: 60,000 training images and 10,000 test images of 28×28 grayscale handwritten digits
(LeCun et al., 1998). The cell below downloads the 4 original files and checks their MD5 hashes, so everyone who
runs it gets exactly the same data. Each image is laid out as a row vector of 784 numbers in $[0, 1]$.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr, ytr = train_img.reshape(-1, 784) / 255.0, train_lab.astype(int)  # each row is one image: x of size 1 × 784
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(int)
print(f"training: {num(len(Xtr), 0)} images {Xtr.shape}  ·  test: {num(len(Xte), 0)} images {Xte.shape}")

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for ax, img, lab in zip(axes, test_img, test_lab):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lab), fontsize=11)
    ax.axis("off")
fmt_axes(plt.gcf())
plt.show()

These are the first 10 test images, each with its true label above it. The first one, a 7, comes back in section 8.

## 3 · Six activation functions and their derivatives

This section turns the functions of sections 0.3, 0.4, 0.6 and 0.7 into code. A neuron computes $z = xW + b$, then
outputs $f(z)$. When it learns by gradient (Lesson 05), the error signal travels backward through the neuron and gets
multiplied by the **derivative** $f'(z)$ (section 0.1). So each function is written together with its derivative:

$$\text{step: } f(z) = [z \ge 0] \qquad \sigma(z) = \frac{1}{1+e^{-z}} \qquad \tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

$$\operatorname{ReLU}(z) = \max(0, z) \qquad \operatorname{GELU}(z) = z\,\Phi(z) \qquad \operatorname{SiLU}(z) = z\,\sigma(z)$$

$\Phi$ is the cumulative distribution function of the standard normal distribution. As GPT-2 and the demo do, we
compute GELU with Hendrycks & Gimpel's approximation $\tfrac12 z\,(1 + \tanh[\sqrt{2/\pi}\,(z + 0.044715\,z^3)])$
(section 7 compares it with the exact form). Sigmoid is written as $\tfrac12(1 + \tanh(z/2))$: the same function, but
it doesn't overflow when $z$ is very negative.

In [ ]:
C_GELU = np.sqrt(2 / np.pi)
NAMES = {
    "linear": "no f",
    "step": "step",
    "sigmoid": "sigmoid",
    "tanh": "tanh",
    "relu": "ReLU",
    "gelu": "GELU",
    "silu": "SiLU",
    "logreg": "softmax 784 → 10",
}


def sigmoid(z):
    return 0.5 * (1 + np.tanh(0.5 * z))  # = 1 / (1 + e^−z)


def act(name, z):
    """Return f(z) and its derivative f′(z), the same shape as z. `name` can also be a function you write yourself
    (section 9)."""
    if callable(name):
        return name(z)
    z = np.asarray(z, float)
    if name == "linear":  # no activation function: f(z) = z
        return z, np.ones_like(z)
    if name == "step":  # the perceptron's function: the derivative is 0 everywhere (undefined at 0)
        return (z >= 0).astype(float), np.zeros_like(z)
    if name == "sigmoid":
        s = sigmoid(z)
        return s, s * (1 - s)
    if name == "tanh":
        t = np.tanh(z)
        return t, 1 - t**2
    if name == "relu":  # convention f′(0) = 0, as in PyTorch and d2l.ai
        return np.maximum(0, z), (z > 0).astype(float)
    if name == "gelu":  # the tanh approximation, as in GPT-2 and the demo
        z2 = z * z
        t = np.tanh(C_GELU * z * (1 + 0.044715 * z2))
        return 0.5 * z * (1 + t), 0.5 * (1 + t) + 0.5 * z * (1 - t * t) * C_GELU * (1 + 3 * 0.044715 * z2)
    if name == "silu":
        s = sigmoid(z)
        return z * s, s * (1 + z * (1 - s))
    raise ValueError(name)

In [ ]:
# Check the derivatives with finite differences: (f(z + ε) − f(z − ε)) / 2ε
z = np.linspace(-6, 6, 1201) + 1e-3  # shifted off z = 0 exactly, where the step function and ReLU break
for name in ["sigmoid", "tanh", "relu", "gelu", "silu"]:
    numeric = (act(name, z + 1e-6)[0] - act(name, z - 1e-6)[0]) / 2e-6
    err = np.abs(numeric - act(name, z)[1]).max()
    assert err < 1e-8, name
    print(f"{NAMES[name]:>7}: largest |f′ − finite difference| = {sci(err)}")

The hand-written derivatives match the finite differences to within a billionth. The cell below prints the
characteristic values that the demo also shows: $f(0)$, $f'(0)$ and the largest derivative (found on a very fine grid
of $z$ from −8 to 8).

In [ ]:
zz = np.linspace(-8, 8, 160_001)
print(f"z grid: {num(len(zz), 0)} points, step {num(zz[1] - zz[0], 4)}\n")
print("   name   f(0)  f′(0)   largest f′ (at z)")
for name in ["sigmoid", "tanh", "relu", "gelu", "silu"]:
    f0, d0 = act(name, 0.0)
    d = act(name, zz)[1]
    k = d.argmax()
    print(f"{NAMES[name]:>7}  {num(f0, 3):>5}  {num(d0, 3):>5}   {num(d[k], 3)} (z = {num(zz[k], 2)})")
assert act("sigmoid", 0.0)[1] == 0.25 and act("tanh", 0.0)[1] == 1.0, "differs from the demo"
assert np.isclose(act("gelu", 0.0)[1], 0.5) and np.isclose(act("silu", 0.0)[1], 0.5)

The figure below plots each function on $z \in [-5, 5]$, as the demo does: $f(z)$ is the solid blue line, $f'(z)$
the dashed orange line. Notice three things: the step function's dashed line lies flat on 0; sigmoid's dashed line
never goes above 0.25; and the dashed lines of GELU and SiLU rise slightly above 1.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5.6), sharex=True)
zp = np.linspace(-5, 5, 1001)
for ax, name in zip(axes.flat, ["step", "sigmoid", "tanh", "relu", "gelu", "silu"]):
    f, d = act(name, zp)
    if name == "step":  # draw the two halves separately, so there is no fake vertical line at z = 0
        ax.plot(zp[zp < 0], f[zp < 0], color=BLUE, lw=2)
        ax.plot(zp[zp >= 0], f[zp >= 0], color=BLUE, lw=2, label="f(z)")
    else:
        ax.plot(zp, f, color=BLUE, lw=2, label="f(z)")
    ax.plot(zp, d, color=ORANGE, lw=2, ls="--", label="f′(z)")
    ax.axhline(0, color=GREY, lw=0.8)
    ax.axvline(0, color=GREY, lw=0.8)
    ax.set_title(NAMES[name])
    ax.set_ylim(-1.3, 2.2 if name in ("relu", "gelu", "silu") else 1.3)
    ax.grid(alpha=0.3)
for ax in axes[1]:
    ax.set_xlabel("z")
axes[0, 0].legend(frameon=False, loc="upper left")
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

## 4 · Without $f$: three layers collapse into one

The course's network is MLSysBook's 784 → 128 → 64 → 10 MLP: $h_1 = f(xW_1 + b_1)$, $h_2 = f(h_1W_2 + b_2)$,
$z = h_2W_3 + b_3$ (10 logits; softmax comes in Lesson 04). The cell below defines the forward pass for any $f$,
with He-initialized weights as in `reel/mnist_mlp/train.py` (Lesson 09 explains why this scale is chosen).

If we drop $f$ (that is, $f(z) = z$), the three matrix multiplications can be merged, as section 0.2 proves:

$$z = \big((xW_1 + b_1)W_2 + b_2\big)W_3 + b_3 = x\,\underbrace{W_1W_2W_3}_{784 \times 10} + \underbrace{(b_1W_2 + b_2)W_3 + b_3}_{1 \times 10}$$

Let's check it numerically: compute both sides on the 10,000 test images, with random weights and nonzero biases.

In [ ]:
SIZES = [784, 128, 64, 10]
BATCH = 64


def he_init(sizes, rng, scale=1.0):
    """He initialization: W ~ N(0, 2/n_in), b = 0 (as in reel/mnist_mlp/train.py). `scale` is used in section 11."""
    return [
        [scale * rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)]
        for n_in, n_out in itertools.pairwise(sizes)
    ]


def forward(f, params, x):
    """Forward pass. hs: the input of each layer, the last element being the 10 logits; ds: f′(z) of each hidden
    layer."""
    hs, ds = [x], []
    for k, (W, b) in enumerate(params):
        z = hs[-1] @ W + b
        if k == len(params) - 1:
            hs.append(z)  # the last layer has no f
        else:
            h, d = act(f, z)
            hs.append(h)
            ds.append(d)
    return hs, ds


rng = np.random.default_rng(0)
params = he_init(SIZES, rng)
for layer in params:
    layer[1] = rng.normal(0, 0.1, layer[1].shape)  # nonzero biases, so that the merge isn't trivial
(W1, b1), (W2, b2), (W3, b3) = params
W_one = W1 @ W2 @ W3
b_one = (b1 @ W2 + b2) @ W3 + b3
z_three = forward("linear", params, Xte)[0][-1]
z_one = Xte @ W_one + b_one
gap = np.abs(z_three - z_one).max()
print(f"merged matrix: {W_one.shape}, bias: {b_one.shape};  largest |z three layers − z one layer| = {sci(gap)}")
macs = sum(n_in * n_out for n_in, n_out in itertools.pairwise(SIZES))
print(f"three layers: {num(macs, 0)} MACs per image  ·  one 784 × 10 matrix: {num(784 * 10, 0)} MACs, the same result")
assert gap < 1e-9 and macs == 109184

The two computations give the same 10 logits, differing only by rounding error ($10^{-15}$). Without $f$, the
three-layer network costs 109,184 multiply-accumulate operations but is only **one** affine transformation
784 → 10, exactly the kind of model of Lesson 01 (one layer, $z = xW + b$). Adding layers without adding $f$ adds
nothing but cost.

What about $f$ = ReLU? The cell below finds the *best* matrix $A$ and bias $c$ (least squares) for $xA + c$ to imitate
the logits of the same random network, this time with ReLU, then measures the part it can't imitate.

In [ ]:
def best_affine_gap(Z, X):
    """The fraction of the variance that no affine map xA + c can explain (0 = it merges completely).
    Z puts several sets of 10 logits side by side as columns; returns one number per set."""
    X1 = np.c_[X, np.ones(len(X))]
    R = Z - X1 @ np.linalg.lstsq(X1, Z, rcond=None)[0]
    return [
        (R[:, k : k + 10] ** 2).sum() / ((Z[:, k : k + 10] - Z[:, k : k + 10].mean(0)) ** 2).sum()
        for k in range(0, Z.shape[1], 10)
    ]


z_relu = forward("relu", params, Xte)[0][-1]
gaps = best_affine_gap(np.c_[z_three, z_relu], Xte)  # one solve for both sets of 10 logits
print(f"no f: {pct(gaps[0], 4)} of the variance left;  ReLU: {pct(gaps[1], 1)}")

Without $f$, one affine map explains everything (0%). With ReLU, the best affine map still leaves 17.6% of the
logits' variance unexplained: the network computes a *non*linear function of the image. That is exactly what makes
depth meaningful.

## 5 · Step → sigmoid: smoothing the switch

Lesson 01's perceptron uses the step function: an on/off switch (section 0.3). Sigmoid is a “soft” version of that
switch (section 0.4). Indeed,
as $T$ shrinks, $\sigma(z/T)$ approaches the step function, and its slope at 0 is $\sigma'(0)/T = 1/(4T)$: steeper,
but narrower. The figure below plots $\sigma(z/T)$ for $T$ = 1, 0.3 and 0.1 (three different line styles) and the
step function (gray).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
zp = np.linspace(-4, 4, 801)
ax.plot(zp[zp < 0], np.zeros((zp < 0).sum()), color=GREY, lw=3, alpha=0.6)
ax.plot(zp[zp >= 0], np.ones((zp >= 0).sum()), color=GREY, lw=3, alpha=0.6, label="step function")
for T, ls, color in [(1, "-", BLUE), (0.3, "--", GREEN), (0.1, ":", ORANGE)]:
    ax.plot(zp, sigmoid(zp / T), color=color, lw=2, ls=ls, label=f"σ(z/T), T = {num(T, 1)}")
    print(f"T = {num(T, 1)}: slope at 0 = {num(act('sigmoid', 0.0)[1] / T, 2)}")
ax.set(xlabel="z", ylabel="output", title="Sigmoid is a smoothed step function")
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.3)
fmt_axes(plt.gcf())
plt.show()

### 🤔 Predict first
Take the first neuron of hidden layer 1 in the random network of section 4 and increase its bias $b$ by 0.001.
On the 10,000 test images, for how many images does the output change if the neuron uses the step function? And if
it uses sigmoid?

In [ ]:
z0 = Xte @ W1[:, 0] + b1[0]  # z of neuron 0 on the 10,000 images
nudge = 0.001
print(f"this neuron's z lies in [{num(z0.min(), 2)}, {num(z0.max(), 2)}]; increase b by {num(nudge, 3)}")
for name in ["step", "sigmoid"]:
    before, after = act(name, z0)[0], act(name, z0 + nudge)[0]
    slope = (after - before) / nudge  # estimated slope: change in output / change in input
    moved = slope[slope != 0]
    print(
        f"{NAMES[name]:>9}: {num((slope == 0).sum(), 0)} images unchanged (slope exactly 0); {num(len(moved), 0)} "
        f"images changed, slope from {num(moved.min(), 0 if name == 'step' else 2)} to "
        f"{num(moved.max(), 0 if name == 'step' else 2)}"
    )

With the step function, nudging $b$ **changes nothing at all** for 9,991 images: the slope is exactly 0. For the 9
images whose $z$ lies right at the threshold, the output jumps all the way from 0 to 1, so the “slope” is
$1/0.001$ = 1,000, and it would become infinite with an even smaller nudge. Neither tells you which way to go.
Gradient descent (Lesson 05) learns by measuring “if I nudge this a little, how much does the loss change”; with the
step function, the answer is 0 almost everywhere. Sigmoid, in contrast, changes the output of *every* image by a
small amount, proportional to $\sigma'(z)$: there is always a direction to move in.

## 6 · Saturation: both ends of the curve are almost flat

Sigmoid and tanh are smooth, but their derivatives shrink toward 0 as $|z|$ grows: $\sigma'(z) = \sigma(z)(1 - \sigma(z))$
and $\tanh'(z) = 1 - \tanh^2(z)$. We call a neuron **saturated** when $f'(z)$ is less than one tenth of its maximum
value. Since $\sigma'(z) = 1/(4\cosh^2(z/2))$ and $\tanh'(z) = 1/\cosh^2 z$, that threshold has an exact solution
(sections 0.4 and 0.5).

In [ ]:
Z_SAT = {  # beyond this |z|, f′(z) < f′(0)/10
    "sigmoid": 2 * np.arccosh(np.sqrt(10)),  # cosh²(z/2) = 10
    "tanh": np.arccosh(np.sqrt(10)),  # cosh²(z) = 10
}
for name, zs in Z_SAT.items():
    assert np.isclose(act(name, zs)[1], act(name, 0.0)[1] / 10)
    print(f"{NAMES[name]:>7} saturates when |z| > {num(zs, 2)}")
print()
print("    z    σ′(z)        tanh′(z)")
for zv in [0, 1, 2, 4, 6, 8]:
    print(f"{zv:>5}    {sci(act('sigmoid', zv)[1], 2):<12} {sci(act('tanh', zv)[1], 2)}")
z = np.linspace(-6, 6, 1201)
assert np.allclose(np.tanh(z), 2 * sigmoid(2 * z) - 1)  # tanh is just sigmoid, rescaled and shifted down
print("\ntanh(z) = 2σ(2z) − 1: holds on the whole z grid")

The two functions differ only by a rescaling: $\tanh(z) = 2\sigma(2z) - 1$. But that rescaling changes two important
things. The derivative of tanh at 0 is 1; that of sigmoid is only 0.25. And tanh's output is centered on 0, while
sigmoid's is always positive. Tanh saturates earlier (from $|z| > 1.82$, against $3.64$ for sigmoid), but its middle
part is 4 times as steep.

The figure below shades each function's saturation region with a hatched gray background. In that region, the dashed
line $f'(z)$ lies almost on the 0 axis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
zp = np.linspace(-6, 6, 1201)
for ax, name in zip(axes, ["sigmoid", "tanh"]):
    f, d = act(name, zp)
    zs = Z_SAT[name]
    for lo, hi in [(-6, -zs), (zs, 6)]:
        ax.axvspan(lo, hi, color=GREY, alpha=0.15, hatch="//", lw=0)
    ax.plot(zp, f, color=BLUE, lw=2, label="f(z)")
    ax.plot(zp, d, color=ORANGE, lw=2, ls="--", label="f′(z)")
    ax.text(4.8, -0.6, "saturated", ha="center", fontsize=10)
    ax.text(-4.8, -0.6, "saturated", ha="center", fontsize=10)
    ax.axhline(0, color=GREY, lw=0.8)
    ax.set(xlabel="z", title=f"{NAMES[name]}: saturates when |z| > {num(zs, 2)}", xlim=(-6, 6))
axes[0].legend(frameon=False, loc="upper left")
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

Why should a small derivative worry us? During backpropagation (Lesson 06), the error signal is multiplied by
$f'(z)$ once at **every** layer. With sigmoid, each multiplication is by at most 0.25, so across $L$ layers the
derivative factors alone are already at most $0.25^L$:

In [ ]:
for L in [1, 2, 5, 10, 20]:
    print(f"{L:>2} sigmoid {'layer: ' if L == 1 else 'layers:'} product of the σ′ ≤ 0.25^{L} = {sci(0.25**L)}")
assert f"{0.25**20:.1e}" == "9.1e-13"  # exactly the number MLSysBook gives for 20 layers

With 20 layers, the upper bound is $9.1 \cdot 10^{-13}$, exactly the number that MLSysBook's *Neural Computation*
chapter gives. And that is the *best* case, when every neuron sits exactly at $z = 0$; saturated neurons make the
signal smaller still. This phenomenon is the **vanishing gradient** problem, the topic of Lesson 09. (The weights $W$
also multiply the signal, so a small product of $f'$ can be partly compensated; Lesson 09 measures that on a real
network.)

### Checking section 0.5: how fast do the derivatives die out?
By section 0.5, $\sigma'(z)\,e^{|z|}$ always lies between $\tfrac14$ and 1, and $\tanh'(z)\,e^{2|z|}$ between 1 and
4: both derivatives die out exponentially in $|z|$. The cell below computes those two ratios on the whole $z$ grid of
section 3, compares them with the exact formulas $1/(1 + e^{-|z|})^2$ and $4/(1 + e^{-2|z|})^2$, then prints by how
much $\sigma'$ and $\tanh'$ shrink when $z$ goes from 7 to 8.

In [ ]:
za = np.abs(zz)
r_sig = act("sigmoid", zz)[1] * np.exp(za)  # σ′(z)·e^|z|
r_tanh = act("tanh", zz)[1] * np.exp(2 * za)  # tanh′(z)·e^2|z|
assert np.allclose(r_sig, 1 / (1 + np.exp(-za)) ** 2, rtol=1e-9, atol=0)
assert np.allclose(r_tanh, 4 / (1 + np.exp(-2 * za)) ** 2, rtol=1e-8, atol=0)
assert 0.25 - 1e-12 <= r_sig.min() and r_sig.max() <= 1 and 1 - 1e-12 <= r_tanh.min() and r_tanh.max() <= 4
print(f"on {num(len(zz), 0)} points z ∈ [−8, 8]:")
print(f"   σ′(z)·e^|z| from {num(r_sig.min(), 6)} to {num(r_sig.max(), 6)}")
print(f"   tanh′(z)·e^2|z| from {num(r_tanh.min(), 6)} to {num(r_tanh.max(), 6)}")
drop_sig, drop_tanh = (act(name, 7.0)[1] / act(name, 8.0)[1] for name in ["sigmoid", "tanh"])
print(
    f"z from 7 to 8: σ′ shrinks {num(drop_sig, 4)} times (e = {num(np.e, 4)}), "
    f"tanh′ shrinks {num(drop_tanh, 4)} times (e² = {num(np.e**2, 4)})"
)

All four bounds of section 0.5 hold on the whole grid, and for large $|z|$ the upper bounds are almost tight: the two
ratios approach 1 and 4. Each extra unit of $|z|$ divides $\sigma'$ by almost $e$ and $\tanh'$ by almost $e^2$, as
the last line shows. So the $f'$ factor of a saturated neuron isn't just smaller than $\tfrac14$: it is
exponentially small in $|z|$.

## 7 · ReLU, GELU and SiLU: simple yet effective

ReLU $= \max(0, z)$ (section 0.6) keeps the positive part as it is and clips the negative part to 0. Its derivative
takes only two values: 1 when $z > 0$ and 0 when $z < 0$ (at exactly $z = 0$ we set it to 0 by convention). The
positive side never saturates, and computing ReLU takes only one comparison.

GELU and SiLU (section 0.7) are two smooth versions of ReLU, both of the form “$z$ times a soft gate in $(0, 1)$”:
GELU multiplies $z$ by $\Phi(z)$, SiLU by $\sigma(z)$. Hendrycks & Gimpel (2016) named both; Elfwing et al. (2017)
used SiLU in reinforcement learning; Ramachandran et al. (2017) rediscovered it with an automated search and called
it Swish, $z\,\sigma(\beta z)$, which with $\beta = 1$ is exactly SiLU. The cell below measures three things: the
lowest point of each function, the accuracy of two approximations of GELU, and their shape near 0.

In [ ]:
erf = np.vectorize(math.erf)
z = np.linspace(-6, 6, 12_001)
gelu_exact = 0.5 * z * (1 + erf(z / np.sqrt(2)))  # z·Φ(z), where Φ is the distribution function of N(0, 1)
print(f"GELU tanh form (GPT-2, demo) − exact form: largest {sci(np.abs(act('gelu', z)[0] - gelu_exact).max())}")
print(f"GELU z·σ(1.702z) form        − exact form: largest {sci(np.abs(z * sigmoid(1.702 * z) - gelu_exact).max())}")
for name in ["gelu", "silu"]:
    f = act(name, zz)[0]
    k = f.argmin()
    print(f"{NAMES[name]:>5}: minimum {num(f[k], 3)} at z = {num(zz[k], 2)}; f(−3) = {num(act(name, -3.0)[0], 3)}")

The tanh form that GPT-2 uses deviates from the exact GELU only by about $10^{-4}$; the form $z\,\sigma(1.702z)$ is
cruder, about $10^{-2}$. Unlike ReLU, GELU and SiLU dip slightly below 0 on the left (lowest at −0.170 and −0.278),
then gradually approach 0.

### Checking section 0.7: SiLU's lowest point is $-W(1/e)$
Section 0.7 derives that SiLU is lowest at $z^* = -1 - W(1/e)$, with a value of exactly $-W(1/e)$, and that its slope
exceeds 1 exactly where $\operatorname{SiLU}(z) > 1$, that is from $z = 1 + W(1/e)$ on. The cell below computes
$W(1/e)$ by iterating $u \leftarrow e^{-1-u}$ (the very equation $u = e^{-1-u}$ of section 0.7), then compares it
with the lowest point that the cell above found on the grid.

In [ ]:
w_e = 0.0
for _ in range(60):  # each iteration multiplies the error by about 0.28 (the slope of e^(−1−u) at the solution)
    w_e = np.exp(-1 - w_e)
z_low, z_up = -1 - w_e, 1 + w_e
(s_low, ds_low), (s_up, ds_up) = act("silu", z_low), act("silu", z_up)
f_s, d_s = act("silu", zz)
k_s = f_s.argmin()
print(f"W(1/e) = {num(w_e, 6)}  (|u·eᵘ − 1/e| = {sci(abs(w_e * np.exp(w_e) - np.exp(-1)))})")
print(f"z* = −1 − W(1/e) = {num(z_low, 6)}: SiLU(z*) = {num(s_low, 6)}, |SiLU′(z*)| = {sci(abs(ds_low))}")
print(
    f"the grid of section 3 (step {num(zz[1] - zz[0], 4)}): lowest {num(f_s[k_s], 6)} at z = {num(zz[k_s], 4)}, "
    f"above SiLU(z*) by {sci(f_s[k_s] - s_low)}"
)
print(f"z = 1 + W(1/e) = {num(z_up, 6)}: SiLU = {num(s_up, 6)}, SiLU′ = {num(ds_up, 6)}")
same = np.array_equal(d_s > 1, f_s > 1)
print(
    f"SiLU′ > 1 at {num((d_s > 1).sum(), 0)} grid points, SiLU > 1 at {num((f_s > 1).sum(), 0)}: "
    f"{'exactly the same points' if same else 'DIFFERENT POINTS'}"
)
assert abs(w_e * np.exp(w_e) - np.exp(-1)) < 1e-15
assert abs(s_low + w_e) < 1e-12 and abs(ds_low) < 1e-12 and abs(s_up - 1) < 1e-12 and abs(ds_up - 1) < 1e-12
assert abs(zz[k_s] - z_low) <= 1e-4 and 0 <= f_s[k_s] - s_low < 1e-8 and same
assert abs(z_low + 1.28) < 0.005 and abs(s_low + 0.278) < 0.0005, "differs from the lesson page: check the data"

Sixty iterations are enough for $u\,e^u$ to match $1/e$ up to rounding error. The lowest point on the grid lies less
than one grid step from $z^*$, and is only about $10^{-10}$ above $-W(1/e)$: around its lowest point, the curve is
as flat as the bottom of a parabola. On the positive side, SiLU and its derivative both equal 1 at $1 + W(1/e)$, and
the two conditions $\operatorname{SiLU}' > 1$ and $\operatorname{SiLU} > 1$ pick out exactly the same grid points:
that is why SiLU's dashed line in the figure of section 3 rises above 1.

The figure below zooms in around 0: ReLU solid, GELU dashed, SiLU dotted.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
zp = np.linspace(-4, 3, 701)
for name, color, ls in [("relu", BLUE, "-"), ("gelu", ORANGE, "--"), ("silu", GREEN, ":")]:
    ax.plot(zp, act(name, zp)[0], color=color, lw=2.2, ls=ls, label=NAMES[name])
ax.axhline(0, color=GREY, lw=0.8)
ax.axvline(0, color=GREY, lw=0.8)
ax.set(xlabel="z", ylabel="f(z)", title="ReLU and its two smooth versions", ylim=(-0.5, 3))
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.3)
fmt_axes(plt.gcf())
plt.show()

The cost differs too: sigmoid, tanh, GELU and SiLU need the exponential function, while ReLU only needs a comparison
with 0. MLSysBook notes that in hardware, exponentials are computed with special instructions, lookup tables or
polynomials, so they are usually more expensive. The cell below times them on your machine, on an array of size
10,000 × 128 (exactly the size of hidden layer 1 when the whole test set goes through the network). The numbers
depend on the machine; look only at the order.

In [ ]:
z_big = np.random.default_rng(1).normal(0, 2, (10_000, 128))
for name in ["relu", "sigmoid", "tanh", "gelu", "silu"]:
    t0 = time.perf_counter()
    for _ in range(5):
        act(name, z_big)
    print(f"{NAMES[name]:>7}: {num((time.perf_counter() - t0) / 5 * 1000, 1)} ms for f and f′")

## 8 · Real $z$ values of the 784 → 128 → 64 → 10 MNIST network

So far we have used $z$ on a ready-made axis. Which $z$ does a real network produce? The cell below retrains exactly
the course's reference network (`reel/mnist_mlp/train.py`): ReLU, He initialization, SGD on mini-batches of 64
images, learning rate $\eta = 0.1$, 5 epochs over the 60,000 images, seed 0, cross-entropy loss on softmax. How the
gradient is computed is the subject of Lessons 05–06; here, only notice one line: the signal going backward through
each hidden layer is **multiplied by $f'(z)$** (the formula of section 0.1).

In [ ]:
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def train(f, eta=0.1, seed=0, n=60_000, epochs=5, sizes=SIZES, scale=1.0):
    """Mini-batch SGD, softmax + cross-entropy, as in reel/mnist_mlp/train.py. Returns the parameters and a copy of
    the initial W."""
    rng = np.random.default_rng(seed)
    params = he_init(sizes, rng, scale)
    W_init = [W.copy() for W, _ in params]
    for _ in range(epochs):
        order = rng.permutation(n)  # the first n training images, reshuffled every epoch
        for i in range(0, n, BATCH):
            idx = order[i : i + BATCH]
            hs, ds = forward(f, params, Xtr[idx])
            dz = softmax(hs[-1])
            dz[np.arange(len(idx)), ytr[idx]] -= 1  # ∂L/∂z of the 10 logits = p − onehot(y) (Lesson 04)
            dz /= len(idx)
            for k in reversed(range(len(params))):  # backpropagation (Lesson 06)
                W, b = params[k]
                gW, gb = hs[k].T @ dz, dz.sum(0)
                if k:
                    dz = (dz @ W.T) * ds[k - 1]  # through the activation function: multiply by f′(z)
                W -= eta * gW
                b -= eta * gb
    return params, W_init


def evaluate(f, params, X=Xte, y=yte):
    z = forward(f, params, X)[0][-1]
    p = softmax(z)
    return float((z.argmax(1) == y).mean()), float(-np.log(np.maximum(p[np.arange(len(y)), y], 1e-300)).mean())

In [ ]:
t0 = time.perf_counter()
ref, _ = train("relu")
acc_ref, loss_ref = evaluate("relu", ref)
n_params = sum(W.size + b.size for W, b in ref)
print(
    f"reference network: {num(n_params, 0)} parameters, {pct(acc_ref)} correct on the test set, loss {num(loss_ref, 4)}"
)
print(f"({num(time.perf_counter() - t0, 1)} s on this machine)")
lin_full, _ = train("linear", sizes=[784, 10])  # the same recipe, but only one affine map 784 → 10 (as in section 4)
print(f"same recipe, softmax regression 784 → 10 (no hidden layer, no f): {pct(evaluate('linear', lin_full)[0])}")
assert n_params == 109386, "differs from MLSysBook"
assert abs(acc_ref - 0.9727) < 0.0005 and abs(loss_ref - 0.0865) < 0.0005, (
    "differs from the reel and the demo: check the data"
)

97.27% on the 10,000 test images, exactly the number in `mnist-mlp.json` that the demos from Lesson 04 on use, and
109,386 parameters, as MLSysBook counts. A single affine map (the same kind of function as the network without $f$
in section 4), trained with the same recipe, reaches only 92.04%: the difference is what the hidden layers and the
activation function bring. Now let's look inside.

### One image through hidden layer 1
Feed the first test image (a 7) into the network. Hidden layer 1 has 128 neurons. After ReLU, how many neurons output
**exactly 0**: almost none, about half, or almost all? Guess before you run it.

In [ ]:
x7 = Xte[0]
z7 = x7 @ ref[0][0] + ref[0][1]  # the 128 values z = xW₁ + b₁ of this image
logits7 = forward("relu", ref, x7[None])[0][-1][0]
print(f"label {yte[0]}; the network predicts {logits7.argmax()} (logit {num(logits7.max(), 3)})")
print(
    f"{(z7 <= 0).sum()}/128 neurons have z ≤ 0, so ReLU outputs exactly 0; z lies in [{num(z7.min())}, {num(z7.max())}]"
)
LOGITS_JSON = [-0.3517, -1.6476, 3.6199, 6.7722, -9.6857, 2.1441, -9.7025, 10.898, -0.8677, 2.6404]  # mnist-mlp.json
assert np.allclose(logits7, LOGITS_JSON, atol=1e-3) and logits7.argmax() == 7, "differs from the reel and the demo"
assert (z7 <= 0).sum() == 70, "differs from the reel"

More than half. Each image “switches on” only some of the neurons; the rest stay completely silent. This is
**sparse activation**, a property that Glorot et al. (2011) consider an advantage of ReLU.

The figure below passes exactly those 128 values of $z$ through five functions. The gray line is the function; each
point is one neuron: a blue circle when $z > 0$, an orange square when $z \le 0$. Note: the network was trained with
ReLU, so the other four panels are a “what if these same $z$ went through another function” test, not a network that
has learned with that function (section 9 trains those).

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 2.6), sharex=True)
zp = np.linspace(z7.min() - 0.3, z7.max() + 0.3, 400)
pos = z7 > 0
for ax, name in zip(axes, ["relu", "sigmoid", "tanh", "gelu", "silu"]):
    ax.plot(zp, act(name, zp)[0], color=GREY, lw=1.5)
    fz = act(name, z7)[0]
    ax.scatter(z7[pos], fz[pos], s=14, color=BLUE, marker="o", label="z > 0")
    ax.scatter(z7[~pos], fz[~pos], s=14, color=ORANGE, marker="s", label="z ≤ 0")
    ax.set_title(NAMES[name], fontsize=10)
    ax.set_xlabel("z")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("f(z)")
axes[0].legend(frameon=False, fontsize=8, loc="upper left")
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

For this image, every $z$ lies well within the range where sigmoid is still steep: no point reaches the flat regions
at either end. The cell below does the same for **all** 10,000 test images and both hidden layers (1,280,000 and
640,000 values of $z$).

In [ ]:
def hidden_z(f, params, X=Xte):
    """z = hW + b of each hidden layer (before the activation function)."""
    hs, _ = forward(f, params, X)
    return [hs[k] @ params[k][0] + params[k][1] for k in range(len(params) - 1)]


zs_ref = hidden_z("relu", ref)
for k, z in enumerate(zs_ref, 1):
    p1, p50, p99 = np.percentile(z, [1, 50, 99])
    print(f"hidden layer {k} ({num(z.size, 0)} values): 98% lie in [{num(p1)}, {num(p99)}], median {num(p50)}")
    print(
        f"   ReLU outputs exactly 0 for {pct((z <= 0).mean(), 1)} of the values; neurons at 0 for all 10,000 images: "
        f"{(z <= 0).all(0).sum()}"
    )
    for name, zsat in Z_SAT.items():
        print(
            f"   if passed through {NAMES[name]}: {pct((np.abs(z) > zsat).mean())} of the values fall in the "
            "saturation region"
        )
print(
    f"the 7 above: {(np.abs(z7) > Z_SAT['sigmoid']).sum()} saturated values with sigmoid, "
    f"{(np.abs(z7) > Z_SAT['tanh']).sum()} with tanh"
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(zs_ref[0].ravel(), bins=np.linspace(-8, 8, 161), color=BLUE, alpha=0.8)
ax.axvline(0, color="black", lw=1)
for name, ls in [("tanh", "--"), ("sigmoid", ":")]:
    for sign in (-1, 1):
        ax.axvline(
            sign * Z_SAT[name],
            color=ORANGE,
            lw=1.8,
            ls=ls,
            label=f"{NAMES[name]} saturation threshold" if sign > 0 else None,
        )
ax.set(
    xlabel="z of hidden layer 1",
    ylabel="number of values",
    title="1,280,000 real z values of hidden layer 1 (10,000 images × 128 neurons)",
    xlim=(-6, 9),  # leave room on the right for the legend
)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: num(v, 0)))
ax.legend(frameon=False, loc="center right")
fmt_axes(plt.gcf())
plt.show()

Three things to remember from the real numbers:
- **No neuron is “dead”.** ReLU outputs 0 for about half of the values, but no neuron is 0 for *every* image. A neuron
  that is 0 for this image still switches on for another, and the gradient still reaches it through those images.
  Only a neuron that is 0 for every input truly stops learning (section 0.6; section 11 creates some).
- **The saturation region is almost empty in layer 1.** Only 0.32% of the $z$ values lie outside sigmoid's two dotted
  lines (7.98% outside tanh's two dashed lines). Saturation doesn't happen by itself; it needs large $z$, from
  initial weights that are too large or a very deep network (section 11, Lesson 09).
- Layer 2's $z$ are spread wider than layer 1's, so tanh would saturate far more there: 27.70%.

## 9 · Change $f$, keep everything else: an experiment

Now we train the same network with each function. Everything else stays the same: He initialization, mini-batches
of 64, $\eta = 0.1$, the same seeds. To fit the run time, each run learns for only **1 epoch on the first 20,000
images** (313 steps), instead of 5 epochs on 60,000; the numbers are therefore lower than the reference network's
97.27%. Each function runs with 3 seeds. There is one more baseline: single-layer softmax regression 784 → 10, that
is, a single affine map.

### 🤔 Predict first
1. With the step function, $f'(z) = 0$, so the gradient never gets through a hidden layer: $W_1$ and $W_2$ stay put.
   Does that network reach about 10% (like random guessing), or clearly more?
2. How does the three-layer network without $f$ compare with single-layer softmax regression: clearly better, clearly
   worse, or about the same?

In [ ]:
N_SHORT, EPOCHS_SHORT = 20_000, 1
FUNCS = ["step", "sigmoid", "linear", "logreg", "tanh", "silu", "gelu", "relu"]
t0 = time.perf_counter()
runs, accs = {}, {}
for f in FUNCS:
    for seed in range(3):
        if f == "logreg":  # one layer 784 → 10, no hidden layer
            runs[f, seed] = train("linear", 0.1, seed, N_SHORT, EPOCHS_SHORT, sizes=[784, 10])
        else:
            runs[f, seed] = train(f, 0.1, seed, N_SHORT, EPOCHS_SHORT)
    accs[f] = [evaluate("linear" if f == "logreg" else f, runs[f, s][0])[0] for s in range(3)]
print(
    f"{len(FUNCS) * 3} training runs, each 1 epoch on {num(N_SHORT, 0)} images = {-(-N_SHORT // BATCH)} steps "
    f"({num(time.perf_counter() - t0, 1)} s)\n"
)


def seed_range(a):
    return f"{pct(min(a))}–{pct(max(a))}"


def verdict(a, b):
    """When the two ranges over the seeds overlap, we can't yet tell which one is better."""
    return (
        "within the spread between seeds"
        if max(min(a), min(b)) <= min(max(a), max(b))
        else "beyond the spread between seeds"
    )


for f in FUNCS:
    print(f"{NAMES[f]:>17}: {' · '.join(pct(a) for a in accs[f])}   (range {seed_range(accs[f])})")
moved = [np.abs(runs["step", s][0][k][0] - runs["step", s][1][k]).max() for s in range(3) for k in range(3)]
print(
    f"\nstep function: largest |W after − W before| of W₁, W₂, W₃ (seed 0) = {', '.join(num(m, 3) for m in moved[:3])}"
)
print(f"random guessing (10 digits): {pct(1 / 10, 0)}")
print(f"no f vs. softmax regression: {verdict(accs['linear'], accs['logreg'])}")
for f in ["tanh", "silu", "gelu", "relu"]:
    print(
        f"{NAMES[f]:>4} vs. no f: {verdict(accs[f], accs['linear'])}; vs. softmax 784 → 10: {verdict(accs[f], accs['logreg'])}"
    )
assert all(m == 0 for k, m in enumerate(moved) if k % 3 < 2), "W₁, W₂ of the step-function network must stay put"

Reading the table above (remember that these are short runs, with the same $\eta = 0.1$ for every function):
- **The step function** doesn't stay at 10%. $W_1$ and $W_2$ do stay put (they change by 0), but $W_3$ still learns,
  because the gradient reaches the last layer without going through any $f'$ (the formula of section 0.1). The
  network becomes a linear classifier on 64 *random* 0/1 features that the first two layers produce at
  initialization. The step function doesn't block all learning; it blocks learning **in the layers before it**.
- **No $f$** gives about the same result as single-layer softmax regression, within the spread between seeds. Three
  linear layers and one layer are the same kind of function (sections 0.2 and 4); they differ only in how gradient
  descent reaches a solution, not in what they can represent.
- **Tanh, GELU and ReLU** beat both linear baselines, beyond the spread between seeds. **SiLU** beats softmax
  regression, but its range overlaps the range of the network without $f$. With these short runs, the gain from $f$
  is only a few percentage points; with the full recipe of section 8, it is 97.27% against 92.04%. The differences
  among these four functions are small.
- **Sigmoid**, though, is clearly worse with the same $\eta = 0.1$. Why?

### 🤔 Predict first
Is sigmoid worse because its neurons saturate, or for another reason? The cell below measures the saturation in the
very sigmoid networks just trained, then tries two hand-written functions, each changing only **one** property of
sigmoid:
- `tanh_quarter` $= \tanh(z)/4$: symmetric around 0 like tanh, but with a slope of only 0.25 at 0, like sigmoid;
- `sigmoid_x4` $= 4\sigma(z) - 2$: sigmoid itself, stretched to be symmetric around 0 with a slope of 1 at 0, like
  tanh.

(`act()` also accepts a Python function that returns $f$ and $f'$, so you can write your own activation function
this way.)

In [ ]:
def tanh_quarter(z):
    t = np.tanh(z)
    return t / 4, (1 - t**2) / 4


def sigmoid_x4(z):
    s = sigmoid(z)
    return 4 * s - 2, 4 * s * (1 - s)


for s in range(3):
    zs = hidden_z("sigmoid", runs["sigmoid", s][0])
    print(
        f"sigmoid network seed {s}: saturated in layer 1 {pct((np.abs(zs[0]) > Z_SAT['sigmoid']).mean(), 3)}, "
        f"layer 2 {pct((np.abs(zs[1]) > Z_SAT['sigmoid']).mean(), 3)}"
    )
print()
for f, label in [
    ("sigmoid", "sigmoid (slope 0.25, always positive)"),
    (tanh_quarter, "tanh/4 (slope 0.25, centered on 0)"),
    (sigmoid_x4, "4σ − 2 (slope 1, centered on 0)"),
    ("tanh", "tanh (slope 1, centered on 0)"),
]:
    a = (
        accs[f]
        if isinstance(f, str)
        else [evaluate(f, train(f, 0.1, s, N_SHORT, EPOCHS_SHORT)[0])[0] for s in range(3)]
    )
    print(f"{label:>37}: {seed_range(a)}")

It isn't saturation: in the trained sigmoid networks, almost no $z$ value lies in the saturation region. The culprit
is the **slope**. The two functions whose derivative at 0 is 0.25 (sigmoid and tanh/4) are both worse; the two with
slope 1 (4σ − 2 and tanh) are almost equal, even though 4σ − 2 has exactly sigmoid's S shape. Each time the signal
goes backward through a sigmoid layer, it is multiplied by at most 0.25, so the early layers get much smaller updates
for the same $\eta$. Section 11 shows that increasing $\eta$ makes up for most of it. Saturation is a real problem,
but it needs large $z$ (section 11) or a deep network (Lesson 09); in a shallow network with He initialization, it
is absent.

## 10 · Sigmoid lives on as a “gate”: SwiGLU

In language models, every Transformer block has a two-layer MLP (Lesson 17). GPT-2 and BERT use
$\operatorname{GELU}(xW_1 + b_1)W_2 + b_2$. Dauphin et al. (2016) proposed the **gated linear unit** (GLU): two
parallel projections, one of which goes through a sigmoid and then multiplies the other element-wise,
$\sigma(xW) \odot xV$. Sigmoid here is not an “activation” but a **gate**: a number in $(0, 1)$ that decides how much
of the signal $xV$ gets through. Shazeer (2020) tried many variants; the one that uses SiLU is called SwiGLU
(section 0.8):

$$\operatorname{FFN}_{\text{SwiGLU}}(x) = \big(\operatorname{SiLU}(xW) \odot xV\big)\,W_2, \qquad \operatorname{SiLU}(u) = u\,\sigma(u)$$

Sigmoid is still in there, inside SiLU. This layer has **three** matrices instead of two, so to keep the number of
parameters the same, Shazeer shrinks the hidden dimension $d_{ff}$ to 2/3. The cell below checks his numbers, then
counts the MLP part of Llama 3 8B from its `config.json`.

In [ ]:
d_model, d_ff = 768, 3072  # T5 base (Shazeer 2020, section 3.1)
ffn_relu = 2 * d_model * d_ff  # W₁, W₂
ffn_glu = 3 * d_model * (2 * d_ff // 3)  # W, V, W₂ with d_ff = 2048
print(f"standard FFN: {num(ffn_relu, 0)} parameters; SwiGLU with d_ff = {2 * d_ff // 3}: {num(ffn_glu, 0)}")
assert ffn_relu == ffn_glu == 4718592

# Llama 3 8B (config.json on Hugging Face): no biases, RMSNorm, output layer not tied to the embedding
L, d, heads, kv_heads, hd, ff, vocab = 32, 4096, 32, 8, 128, 14336, 128256
attn = L * (d * (heads * hd + 2 * kv_heads * hd) + heads * hd * d)
mlp = L * 3 * d * ff  # per layer: W, V of size d × ff and W₂ of size ff × d
total = vocab * d + attn + mlp + (2 * L + 1) * d + vocab * d
print(f"Llama 3 8B: {num(total, 0)} parameters; the SwiGLU MLPs account for {num(mlp, 0)} ({pct(mlp / total, 1)})")
assert total == 8030261248

CONFIGS = {  # the MLP's activation function and hidden dimension, read from each model's config.json
    "GPT-2 (2019)": ("gelu_new = GELU, tanh form", 768, 3072),
    "BERT base (2018)": ("gelu", 768, 3072),
    "Llama 3 8B (2024)": ("silu, gated = SwiGLU", 4096, 14336),
    "Qwen3-8B (2025)": ("silu, gated = SwiGLU", 4096, 12288),
    "DeepSeek-V3 (2024)": ("silu, gated, per expert", 7168, 2048),  # 256 small experts (Lesson 24)
}
for name, (fn, dm, dff) in CONFIGS.items():
    print(f"{name:>19}: {fn:<29} d = {num(dm, 0):>5}, d_ff = {num(dff, 0):>6} ({num(dff / dm, 2)} × d)")

Shazeer chooses $d_{ff} = 2048$ instead of 3072, so the two options have exactly the same number of parameters. In
Llama 3 8B, the SwiGLU MLPs make up more than two thirds of the whole model: the activation function sits in the
largest part of the network. GPT-2 and BERT use $d_{ff} = 4d$; the models that use SwiGLU have a correspondingly
smaller hidden dimension (Llama 3: 3.5 × d), because of the extra third matrix.

Now we build a real SwiGLU layer in place of hidden layer 1 of the MNIST network: $h_1 = \operatorname{SiLU}(xW) \odot xV$
with $W, V$ of size 784 × 64. Those two matrices have exactly $2 \cdot 784 \cdot 64$ = 100,352 weights, as many as
the original network's 784 × 128 $W_1$. The rest keeps ReLU: 64 → 64 → 10. Before training, we check the
hand-written gradient (the gradient through the gate of section 0.8) with finite differences.

In [ ]:
def glu_forward(params, x):
    (W, V), (W2, b2), (W3, b3) = params
    u, v = x @ W, x @ V
    g, dg = act("silu", u)
    h1 = g * v  # SiLU(xW) ⊙ xV: a gated layer, without biases, as in Shazeer
    z2 = h1 @ W2 + b2
    h2 = np.maximum(0, z2)
    return (u, v, g, dg, h1, z2, h2), h2 @ W3 + b3


def glu_grads(params, x, y):
    (_, v, g, dg, h1, z2, h2), z = glu_forward(params, x)
    dz = softmax(z)
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)
    W2, W3 = params[1][0], params[2][0]
    dh2 = (dz @ W3.T) * (z2 > 0)
    dh1 = dh2 @ W2.T
    # h1 = g ⊙ v: the gradient splits into two branches; the branch through the gate is also multiplied by SiLU′(u)
    return [x.T @ (dh1 * v * dg), x.T @ (dh1 * g), h1.T @ dh2, dh2.sum(0), h2.T @ dz, dz.sum(0)]


def glu_init(rng, h=64):
    W, V = (rng.normal(0, np.sqrt(2 / 784), (784, h)) for _ in range(2))
    return [[W, V], *he_init([h, 64, 10], rng)]


def glu_loss(params, x, y):
    p = softmax(glu_forward(params, x)[1])
    return -np.log(p[np.arange(len(y)), y]).mean()

In [ ]:
rng = np.random.default_rng(5)  # a small SwiGLU network (h = 8) to check the gradient
P = glu_init(rng, h=8)
x, y = Xtr[:16], ytr[:16]
worst = 0.0
for arr, g in zip([a for layer in P for a in layer], glu_grads(P, x, y)):
    for _ in range(5):  # 5 random elements of each array
        i = tuple(rng.integers(0, n) for n in arr.shape)
        old = arr[i]
        arr[i] = old + 1e-6
        up = glu_loss(P, x, y)
        arr[i] = old - 1e-6
        down = glu_loss(P, x, y)
        arr[i] = old
        worst = max(worst, abs((up - down) / 2e-6 - g[i]))
print(f"largest |hand-written gradient − finite difference|: {sci(worst)}")
assert worst < 1e-7

In [ ]:
def train_glu(eta=0.1, seed=0, n=N_SHORT):
    rng = np.random.default_rng(seed)
    params = glu_init(rng)
    order = rng.permutation(n)
    for i in range(0, n, BATCH):  # 1 epoch on n images, as in section 9
        idx = order[i : i + BATCH]
        for p, g in zip([a for layer in params for a in layer], glu_grads(params, Xtr[idx], ytr[idx])):
            p -= eta * g
    return params


acc_glu = []
for s in range(3):
    P = train_glu(0.1, s)
    acc_glu.append(float((glu_forward(P, Xte)[1].argmax(1) == yte).mean()))
n_glu = sum(a.size for layer in P for a in layer)
n_wv = P[0][0].size + P[0][1].size
assert n_wv == 784 * 128  # W, V of size 784 × 64: exactly as many weights as the original network's 784 × 128 W₁
print(f"W, V: {num(n_wv, 0)} weights, as many as the original network's 784 × 128 W₁")
print(f"SwiGLU ({num(n_glu, 0)} parameters): {' · '.join(pct(a) for a in acc_glu)}  (range {seed_range(acc_glu)})")
print(
    f"ReLU   ({num(n_params, 0)} parameters): {' · '.join(pct(a) for a in accs['relu'])}  (range {seed_range(accs['relu'])})"
)
print(f"SwiGLU vs. ReLU: {verdict(acc_glu, accs['relu'])}")
SHAZEER = {"ReLU": (1.997, 0.005), "GELU": (1.983, 0.005), "GEGLU": (1.942, 0.004), "SwiGLU": (1.944, 0.010)}
print(f"\nShazeer (2020), table 1: T5's log-perplexity after {num(65536, 0)} steps (standard deviation over 4 runs)")
for name, (lp, sd) in SHAZEER.items():
    print(f"   FFN {name:<6}: {num(lp, 3)} ({num(sd, 3)})")

On this small problem, SwiGLU is **not** clearly better: its range of results overlaps ReLU's, that is, it is within
the spread between seeds, and one of its seeds is clearly worse. That doesn't contradict Shazeer. The gain he
measured is on a T5 model with hundreds of thousands of training steps: the log-perplexity drops from 1.997 (ReLU)
to 1.944 (SwiGLU) after 65,536 steps, while the standard deviation between runs is only 0.005 and 0.010. And he
himself writes that the paper offers no explanation for why these architectures work well, attributing their success
instead to “divine benevolence”. A choice the whole field uses (Llama 3, Qwen3, DeepSeek-V3) can still be an
empirical result rather than a theorem.

## 11 · Try changing things

The cells below rerun the same `train()` function with other settings. You can change the values and run them
again. Each line prints the test accuracy, together with the number of **dead** neurons (0 for all 10,000 test images)
for ReLU, or the fraction of $z$ in the saturation region for sigmoid and tanh. Three experiments:
- sigmoid with $\eta = 1$ instead of 0.1: making up for the small slope;
- ReLU with $\eta = 1$: steps that are too large;
- tanh with initial weights 10 times larger: large $z$ from the start.

In [ ]:
with np.errstate(over="ignore", invalid="ignore"):  # too large an η can make the network diverge (prints "nan")
    for f, eta, scale in [("sigmoid", 1.0, 1), ("relu", 1.0, 1), ("tanh", 0.1, 10)]:
        print(f"{NAMES[f]}, η = {num(eta, 1)}, initialization × {scale}:")
        for seed in range(3):
            p, _ = train(f, eta, seed, N_SHORT, EPOCHS_SHORT, scale=scale)
            zs = hidden_z(f, p)
            if f == "relu":  # dead neurons: z ≤ 0 for all 10,000 images
                dead = [int((z <= 0).all(0).sum()) for z in zs]
                inside = f"dead neurons {dead[0]}/128 ({pct(dead[0] / 128, 1)}), {dead[1]}/64"
            else:  # saturation by the function's own threshold
                sat = [pct((np.abs(z) > Z_SAT[f]).mean(), 1) for z in zs]
                inside = f"saturated (|z| > {num(Z_SAT[f])}) layer 1: {sat[0]}, layer 2: {sat[1]}"
            print(f"   seed {seed}: {pct(evaluate(f, p)[0])} correct; {inside}")

- **Sigmoid with $\eta = 1$** does far better than at $\eta = 0.1$: a slope of 0.25 just needs a larger learning
  step. The larger weights also push a few $z$ into the saturation region (4.4–5.6% in layer 1), but they remain a
  minority. Comparing two activation functions at the *same* $\eta$ compares both the functions and how well $\eta$
  suits each one; a fair experiment picks $\eta$ separately for each function on a validation set and trains long
  enough (Lesson 07 picks $\eta$ that way), so its numbers differ from the short runs of section 9.
- **ReLU with $\eta = 1$** kills many neurons of layer 1 (section 0.6). A step that is too large pushes the bias and
  weights to where $z \le 0$ for every image; from then on $f'(z) = 0$ for every image, the gradient never arrives
  again, and the neuron can't come back to life. The CS231n notes warn about exactly this: as much as 40% of a network
  can die when the learning rate is too high. Here it is more than a third of layer 1 for every seed, and the
  accuracy both drops and swings widely from seed to seed. With steps this large, training is also sensitive to
  rounding error: on another machine (such as Colab), each seed's accuracy and number of dead neurons can differ from
  the numbers printed here, sometimes by a lot, but the conclusion stays the same.
- **Tanh with initialization × 10** pushes $z$ into the saturation region (section 0.5) and keeps them there: most
  $z$ values have $|z| > 1.82$ after training, and the accuracy falls far behind. This is real saturation, which the
  He-initialized networks of section 9 don't run into. Lesson 09 studies why the scale of the initialization matters
  so much.

### Checking section 0.6: dead neurons get a gradient of exactly 0
Section 0.6 proves that a neuron with $z \le 0$ for every training image gets a gradient of exactly 0, for both its
incoming and its outgoing weights, so SGD never changes it again. The cell below reruns exactly the ReLU run with
$\eta = 1$, seed 0, from above, counts the layer-1 neurons that are dead on the very 20,000 images that run learned
from, then computes the gradient over all 20,000 of them with the formula of section 0.1:
$\delta_2 = (\delta_3 W_3^\top) \odot f'(z_2)$ and $\delta_1 = (\delta_2 W_2^\top) \odot f'(z_1)$. It prints the
number of dead neurons, their largest gradient, and the smallest gradient among the living ones.

In [ ]:
p_dead, _ = train("relu", 1.0, 0, N_SHORT, EPOCHS_SHORT)  # exactly the seed-0 run above
X_seen, y_seen = Xtr[:N_SHORT], ytr[:N_SHORT]  # the images that run used
hs_d, ds_d = forward("relu", p_dead, X_seen)
dead1 = (ds_d[0] == 0).all(0)  # f′(z) = 0, that is z ≤ 0, for every training image
delta = softmax(hs_d[-1])  # δ₃ = ∂L/∂z₃ of the 10 logits (Lesson 04), averaged over all 20,000 images
delta[np.arange(N_SHORT), y_seen] -= 1
delta /= N_SHORT
delta2 = (delta @ p_dead[2][0].T) * ds_d[1]  # δ₂ = (δ₃ W₃ᵀ) ⊙ f′(z₂)
delta1 = (delta2 @ p_dead[1][0].T) * ds_d[0]  # δ₁ = (δ₂ W₂ᵀ) ⊙ f′(z₁)
g_in, g_bias, g_out = X_seen.T @ delta1, delta1.sum(0), hs_d[1].T @ delta2  # ∂L/∂W₁, ∂L/∂b₁, ∂L/∂W₂
dead_test = (hidden_z("relu", p_dead)[0] <= 0).all(0)
print(
    f"layer 1: {dead1.sum()}/128 neurons dead on the {num(N_SHORT, 0)} training images; "
    f"{dead_test.sum()}/128 at 0 for all 10,000 test images (the count above)"
)
print(
    f"dead neurons: δ₁ nonzero for {(delta1[:, dead1] != 0).sum()} of {num(delta1[:, dead1].size, 0)} (image, neuron) "
    f"pairs; largest |gradient| of their W₁ columns, b₁ entries and W₂ rows: {num(np.abs(g_in[:, dead1]).max(), 0)}, "
    f"{num(np.abs(g_bias[dead1]).max(), 0)}, {num(np.abs(g_out[dead1]).max(), 0)}"
)
print(
    f"{(~dead1).sum()} living neurons: every W₁ column has a nonzero gradient "
    f"(largest |gradient| of the weakest column: {sci(np.abs(g_in[:, ~dead1]).max(0).min())})"
)
assert dead1.any() and np.all(delta1[:, dead1] == 0) and np.all(g_bias[dead1] == 0) and np.all(g_out[dead1] == 0)
assert np.array_equal(np.abs(g_in).max(0) == 0, dead1)  # the gradient is 0 exactly at the dead neurons, nowhere else

Each dead neuron gets a gradient of **exactly 0**, not just close to 0: its $\delta_1$ is 0 for every training
image, so every mini-batch leaves its $W_1$ column, its bias in $b_1$ and its $W_2$ row as they are. Since the input
of layer 1 is the images themselves, its $z$ stays the same too, and it is dead for good. Conversely, the gradient is
0 *only* at the dead neurons: every living neuron still gets a nonzero gradient. The two counts need not agree: the
cell above counts on the test images, but what decides learning is the training images. A neuron that is silent for
every test image can still switch on for a few training images, and still learn.

## 12 · Limits and next steps

We have chosen $f$ and seen that it decides what the network can learn, but we have only taken one line out of
backpropagation, the line that multiplies by $f'(z)$ at every layer (section 0.1; Lessons 05–06 derive it in full),
and we haven't yet seen what a hidden layer does to the data.
The smallest example is XOR, the problem that Lesson 01 showed a single neuron can't solve. The book *Deep Learning*
(Goodfellow et al., section 6.1) gives a solution with ReLU and two hidden neurons:
$W = \begin{bmatrix}1 & 1\\ 1 & 1\end{bmatrix}$, $c = (0, -1)$, $w = (1, -2)^\top$, $b = 0$.
The cell below runs exactly those weights, with and without ReLU.

In [ ]:
P4 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
W_xor, c_xor, w_xor = np.array([[1.0, 1.0], [1.0, 1.0]]), np.array([0.0, -1.0]), np.array([1.0, -2.0])
for f in ["relu", "linear"]:
    out = act(f, P4 @ W_xor + c_xor)[0] @ w_xor
    print(f"{NAMES[f]:>10}: outputs for (0,0), (0,1), (1,0), (1,1) = {[num(o, 0) for o in out]}")
assert list(act("relu", P4 @ W_xor + c_xor)[0] @ w_xor) == [0, 1, 1, 0]

With ReLU, the network computes XOR exactly: 0, 1, 1, 0. Without ReLU, the same weights reduce to
$2 - (x_1 + x_2)$ (section 0.2), which gives 2, 1, 1, 0, and no linear function can output 0, 1, 1, 0. Lesson 03
unpacks this solution: the hidden layer maps the four points into a new space where a straight line separates them.

## Summary
- Without $f$, the three layers 784 → 128 → 64 → 10 collapse into one 784 × 10 matrix (discrepancy $10^{-15}$), and
  learn about as well as single-layer softmax regression.
- The step function has $f' = 0$: a small nudge changes nothing, so the layers before it stand still; only the last
  layer can learn.
- $\sigma'(0) = 0.25$ and $\tanh'(0) = 1$; across 20 sigmoid layers, the product of the $\sigma'$ is at most
  $9.1 \cdot 10^{-13}$. Moreover $\sigma'(z) \le e^{-|z|}$ and $\tanh'(z) \le 4e^{-2|z|}$: at both ends, the
  derivatives die out exponentially.
- GELU and SiLU are $z$ times a soft gate; SiLU's lowest value is exactly $-W(1/e)$, at $z = -1 - W(1/e)$.
- In the real MNIST network (97.27%), one image switches off 70/128 ReLU neurons in layer 1, but no neuron is dead;
  sigmoid's saturation region is almost empty.
- Sigmoid learns slowly in a shallow network because of its small slope, not because of saturation; ReLU with too
  large an $\eta$ dies, and tanh with too large an initialization saturates.
- A dead ReLU neuron ($z \le 0$ for every training image) gets a gradient of exactly 0 for both its incoming and its
  outgoing weights; in layer 1 it never comes back.
- SwiGLU $= (\operatorname{SiLU}(xW) \odot xV)W_2$ accounts for more than two thirds of Llama 3 8B's parameters; on
  small MNIST, its gain is within the spread between seeds.

## Review questions
1. Why is a three-layer network without an activation function only an affine map? What size is the merged matrix?
2. The network with the step function still gets about half of the images right even though $W_1$, $W_2$ don't
   change. Which layer learned, and from what?
3. What are $\sigma'(0)$ and $\tanh'(0)$? Why does their product across many layers matter?
4. With $\eta = 0.1$, the sigmoid network does worse than the tanh network. What evidence in the notebook shows that
   the reason isn't saturation?
5. How is a ReLU neuron that “outputs 0 for this image” different from a “dead” neuron? What creates dead neurons?
6. How many weight matrices does a SwiGLU layer have? Why does Shazeer reduce $d_{ff}$ from 3072 to 2048?

<details><summary>Hints and answers</summary>

1. $((xW_1 + b_1)W_2 + b_2)W_3 + b_3 = x(W_1W_2W_3) + \text{constant}$; the merged matrix is 784 × 10 (section 0.2).
2. The last layer, $W_3$: the gradient reaches it without going through $f'$. It learns a linear classifier on the
   64 random 0/1 features produced by the first two layers (frozen since initialization).
3. 0.25 and 1. Backpropagation multiplies the signal by $f'(z)$ at every layer, so factors smaller than 1 make the
   signal shrink exponentially (Lesson 09).
4. The fraction of saturated $z$ in the trained sigmoid networks is almost 0; and tanh/4 (slope 0.25) is as bad as
   sigmoid, while 4σ − 2 (sigmoid's shape, slope 1) almost matches tanh. Increasing $\eta$ also makes up for most
   of the gap.
5. Outputting 0 for one image is normal (about half of the values); that neuron still learns from the images that
   switch it on. A dead neuron has $z \le 0$ for *every* image, so $f'(z) = 0$ always and it never learns
   again. Too large a learning rate creates them. A dead neuron's gradient is exactly 0 for both its incoming and
   its outgoing weights (section 0.6; the check in section 11).
6. Three ($W$, $V$, $W_2$) instead of two. $3 \cdot 768 \cdot 2048 = 2 \cdot 768 \cdot 3072$, so the comparison
   keeps the number of parameters and the amount of computation the same.
</details>

## Further reading
- MLSysBook, [*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html): the MNIST network,
  activation functions and their hardware cost.
- V. Nair & G. Hinton (2010), [Rectified Linear Units Improve Restricted Boltzmann Machines](https://www.cs.toronto.edu/~hinton/absps/reluICML.pdf).
- X. Glorot, A. Bordes & Y. Bengio (2011), [Deep Sparse Rectifier Neural Networks](https://proceedings.mlr.press/v15/glorot11a.html): ReLU and sparse activation.
- D. Hendrycks & K. Gimpel (2016), [Gaussian Error Linear Units (GELUs)](https://arxiv.org/abs/1606.08415): GELU, its two approximations, and the name SiLU.
- S. Elfwing, E. Uchibe & K. Doya (2017), [Sigmoid-Weighted Linear Units](https://arxiv.org/abs/1702.03118); P. Ramachandran, B. Zoph & Q. Le (2017), [Searching for Activation Functions](https://arxiv.org/abs/1710.05941) (Swish).
- Y. Dauphin et al. (2016), [Language Modeling with Gated Convolutional Networks](https://arxiv.org/abs/1612.08083) (GLU); N. Shazeer (2020), [GLU Variants Improve Transformer](https://arxiv.org/abs/2002.05202) (SwiGLU).
- The CS231n notes, [Neural Networks Part 1](https://cs231n.github.io/neural-networks-1/): sigmoid, tanh, ReLU and dead neurons.
- [Dive into Deep Learning, section 5.1](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html) and [Deep Learning, chapter 6](https://www.deeplearningbook.org/contents/mlp.html).
- A. Karpathy, [Building makemore Part 3: Activations & Gradients, BatchNorm](https://youtu.be/P6sfmUTpUmc): tanh saturating in a real network.

**Next lesson:** [Lesson 03 · Multilayer networks & the XOR problem](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/mlp-va-xor/): the hidden layer maps the data into a new space where XOR becomes separable.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")